In [1]:
import pandas as pd
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

PROCESSED_PATH = "../data/processed"
df = pd.read_csv(f"{PROCESSED_PATH}/attrition_features.csv")

y = df['Attrition_Label']
X = pd.get_dummies(df.drop(columns=['Attrition_Label', 'EmployeeNumber'], errors='ignore'), drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Define the models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = []
best_model = None
best_auc = 0

print("Running the ultimate ML showdown...\n")

for name, model in models.items():
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    
    # Evaluate
    auc = roc_auc_score(y_test, probs)
    results.append({
        "Model": name,
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1-Score": f1_score(y_test, preds),
        "ROC-AUC": auc
    })
    
    # Track the winner based on ROC-AUC
    if auc > best_auc:
        best_auc = auc
        best_model = model

# Display the comparison table
results_df = pd.DataFrame(results).sort_values(by="ROC-AUC", ascending=False)
print(results_df.to_string(index=False))

# Tech magic: Automatically save the winning pipeline
os.makedirs("../models", exist_ok=True)
model_path = "../models/attrition_pipeline.joblib"
joblib.dump(best_model, model_path)
print(f"\nWinning model successfully saved to: {model_path}")

Running the ultimate ML showdown...



c:\Users\acer\Desktop\Enterprise_Hr_ai\env\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              Model  Precision   Recall  F1-Score  ROC-AUC
      Random Forest   0.363636 0.085106  0.137931 0.796839
Logistic Regression   0.866667 0.276596  0.419355 0.766302
            XGBoost   0.684211 0.276596  0.393939 0.745456

Winning model successfully saved to: ../models/attrition_pipeline.joblib


c:\Users\acer\Desktop\Enterprise_Hr_ai\env\Lib\site-packages\xgboost\training.py:200: UserWarning: [20:32:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
